In [ ]:
from quspin.basis import spinful_fermion_basis_1d # 用于创建一维1/2自旋费米子链的希尔伯特空间基；
from quspin.operators import hamiltonian # 用于在给定的基（basis）上构建哈密顿量算符(或其他物理观测量)；
from Physical_Model_on_Honeycomb import tJ_Model_honeycomb_8site
import numpy as np 
import matplotlib.pyplot as plt  # 用于结果可视化
from matplotlib.ticker import MultipleLocator # MultipleLocator是matplotlib中的一个刻度定位器类,用来按照指定的倍数来设置坐标轴刻度
import ast # 导入Python的ast（抽象语法树）模块，用于安全地解析字符串形式的Python数据结构

In [ ]:
#---------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------########### t-J Model on honeycomb(8格点) ################----------------------------------------
# 参数设置
L = 8 # 格点数
hole_doping = 2/8 # 空穴率
t1 = 1.0 # 最近邻跃迁项(hopping)系数；
J1 = 0.2 # 最近邻相互作用系数；

t2 = 0.0 # 次近邻跃迁项(hopping)系数；
J2 = 0.0 # 次近邻相互作用系数；

#### 哈密顿量的构建
H, basis = tJ_Model_honeycomb_8site(hole_doping, t1, J1, t2, J2)

print('='*80)
print('t-J Model的严格对角化')
print()

## 查看H的特性
print(f"系统尺寸 L = {L}")
print("H 对象的类型:", type(H))
print("希尔伯特空间维度:", H.Ns) # 通过H对象访问其basis的属性
print("矩阵形状:", H.shape) # H作为运算符的矩阵形状s

####### 具体计算
#### 1. 基态能量
E_gs, V_gs = H.eigsh(k=1, which='SA') # eigsh为稀疏矩阵对角化法，k参数要求方法计算前k个本征值，which='SA'代表取前k个最小本征值与本征态
E_gs = E_gs[0] # 因为返回的E_gs是数组(哪怕只有一个元素)，所以需通过E_gs[0]得到具体数字
V_gs = V_gs[:, 0] # 注意：psi_gs是形状为(Ns,1)的二维矩阵(Ns为希尔伯特空间的维数),而psi_gs[:,0]是形状为(Ns,)的一维数组(以后计算以数组为主)

#### 2. 自旋Sz算符平均值
## 计算每个格点的自旋基态平均值<S^z_i>，其中第i格点的Sz_i = 1/2 * (𝑐†_𝑖,↑·𝑐_𝑖,↑ - 𝑐†_𝑖,↓·𝑐_𝑖,↓)
Sz_ED = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_Sz = [
        ["n|", [[0.5, i]]],  # 1/2 * 𝑐†_𝑖,↑·𝑐_𝑖,↑
        ["|n", [[-0.5, i]]] # -1/2 * 𝑐†_𝑖,↓·𝑐_𝑖,↓
    ] 
    dynamic_Sz = [] 
    S_z_i = hamiltonian(static_Sz, dynamic_Sz, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值
    Sz_i = S_z_i.expt_value(V_gs).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    Sz_ED.append(Sz_i)


#### 3. 密度算符平均值
## 计算每个格点的密度基态平均值<n_i>，其中n_i = n_i,↑ + n_i,↓
n_ED = []  # 创建存储每个格点的密度n平均值

for i in range(L):
    # 构建第i个格点的n算符(与哈密顿量的构建完全类似)
    static_n = [
        ["n|", [[1.0, i]]],  # n_i,↑
        ["|n", [[1.0, i]]]   # n_i,↓
    ] 
    dynamic_n = [] 
    ni = hamiltonian(static_n, dynamic_n, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值
    n_i = ni.expt_value(V_gs).real
    n_ED.append(n_i)

#### 将以上测量值打印
print("\n============= 测量结果 ==============")
print(f"\nt-J Model基态能量: {E_gs:.15f}")
print("\n-------- 格点期望值 ---------")
print(f"{'Site':<17} {'<Sz>':<25} {'<n>':<20}")
for i in range(L):
    print(f"{i+1:<10} {Sz_ED[i]:<25.15f} {n_ED[i]:<20.15f}")


In [ ]:
# 1. 将哈密顿量转换为稠密矩阵并直接对角化
H_dense = H.todense()  # 将稀疏矩阵转换为稠密矩阵
eigenvalues, eigenvectors = np.linalg.eigh(H_dense)  # 完整对角化

print("="*80)
print("完整对角化结果")
print(f"总本征值数量: {len(eigenvalues)}")
print(f"希尔伯特空间维度: {H.Ns}")

# 打印所有本征值
print("\n所有本征值 (从低到高):")
for i, eig in enumerate(eigenvalues):
    print(f"E[{i}] = {eig:.10f}")

# 检查基态简并
print("\n基态分析:")
print(f"基态能量 E_gs = {eigenvalues[0]:.10f}")

# 查找与基态能量简并的状态
tolerance = 1e-10  # 能量比较的容差
degenerate_states = []
for i, eig in enumerate(eigenvalues):
    if abs(eig - eigenvalues[0]) < tolerance:
        degenerate_states.append(i)

print(f"基态简并度: {len(degenerate_states)}")
if len(degenerate_states) > 1:
    print(f"基态是简并的! 简并的本征值索引: {degenerate_states}")
    print(f"这些状态的能量: {eigenvalues[degenerate_states]}")
else:
    print("基态是非简并的")

# 计算能隙
if len(eigenvalues) > 1:
    gap = eigenvalues[1] - eigenvalues[0]
    print(f"基态与第一激发态能隙: {gap:.10f}")

In [ ]:
#### 取两个简并基态
H_dense = H.todense()  # 将稀疏矩阵转换为稠密矩阵
eigenvalues, eigenvectors = np.linalg.eigh(H_dense)  # 完整对角化
eigenvectors = np.asarray(eigenvectors)  # eigenvectors是np.matrix对象,需要转换成np.ndarray对象来进行提取,这样可以保证2、3行代码为一维数组
V_gs_1 = eigenvectors[:, 0]
V_gs_2 = eigenvectors[:, 1]


################################################ 第一个简并基态的物理量计算 ##############################################
## 1.自旋Sz算符平均值
## 计算每个格点的自旋基态平均值<S^z_i>，其中第i格点的Sz_i = 1/2 * (𝑐†_𝑖,↑·𝑐_𝑖,↑ - 𝑐†_𝑖,↓·𝑐_𝑖,↓)
Sz_ED_1 = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_Sz = [
        ["n|", [[0.5, i]]],  # 1/2 * 𝑐†_𝑖,↑·𝑐_𝑖,↑
        ["|n", [[-0.5, i]]] # -1/2 * 𝑐†_𝑖,↓·𝑐_𝑖,↓
    ] 
    dynamic_Sz = [] 
    S_z_i = hamiltonian(static_Sz, dynamic_Sz, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    Sz_i = S_z_i.expt_value(V_gs_1).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    Sz_ED_1.append(Sz_i)


## 密度算符平均值
## 计算每个格点的密度基态平均值<n_i>，其中n_i = n_i,↑ + n_i,↓
n_ED_1 = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_n = [
        ["n|", [[1.0, i]]],  # n_i,↑
        ["|n", [[1.0, i]]]   # n_i,↓
    ] 
    dynamic_n = [] 
    ni = hamiltonian(static_n, dynamic_n, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    n_i = ni.expt_value(V_gs_1).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    n_ED_1.append(n_i)

#### 将以上测量值打印
print("\n============= 第一个简并基态的物理量 ==============")
print(f"{'Site':<17} {'<Sz>':<25} {'<n>':<20}")
for i in range(L):
    print(f"{i:<10} {Sz_ED_1[i]:<25.15f} {n_ED_1[i]:<20.15f}")


################################################ 第二个简并基态的物理量计算 ##############################################
## 1.自旋Sz算符平均值
## 计算每个格点的自旋基态平均值<S^z_i>，其中第i格点的Sz_i = 1/2 * (𝑐†_𝑖,↑·𝑐_𝑖,↑ - 𝑐†_𝑖,↓·𝑐_𝑖,↓)
Sz_ED_2 = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_Sz = [
        ["n|", [[0.5, i]]],  # 1/2 * 𝑐†_𝑖,↑·𝑐_𝑖,↑
        ["|n", [[-0.5, i]]] # -1/2 * 𝑐†_𝑖,↓·𝑐_𝑖,↓
    ] 
    dynamic_Sz = [] 
    S_z_i = hamiltonian(static_Sz, dynamic_Sz, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    Sz_i = S_z_i.expt_value(V_gs_2).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    Sz_ED_2.append(Sz_i)


## 密度算符平均值
## 计算每个格点的密度基态平均值<n_i>，其中n_i = n_i,↑ + n_i,↓
n_ED_2 = []  # 创建存储每个格点的S_z平均值

for i in range(L):
    # 构建第i个格点的Sz算符(与哈密顿量的构建完全类似)
    static_n = [
        ["n|", [[1.0, i]]],  # n_i,↑
        ["|n", [[1.0, i]]]   # n_i,↓
    ] 
    dynamic_n = [] 
    ni = hamiltonian(static_n, dynamic_n, basis=basis, check_symm=False, check_pcon=False, check_herm=False)

    # 计算基态期望值，并转换为自旋算符S_z（除以2）
    n_i = ni.expt_value(V_gs_2).real  # S_z_i.expt_value为S_z_i的期望值计算函数,V_gs为上面得到的基态,real表示期望值为实数
    n_ED_2.append(n_i)

#### 将以上测量值打印
print("\n============= 第二个简并基态的物理量 ==============")
print(f"{'Site':<17} {'<Sz>':<25} {'<n>':<20}")
for i in range(L):
    print(f"{i:<10} {Sz_ED_2[i]:<25.15f} {n_ED_2[i]:<20.15f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches # patches模块：用于绘制各种几何形状，并可以控制图形的填充颜色、边框、透明度等属性

##### 1.各格点的密度数据
densities = n_ED_1

##### 2.八格点honeycomb晶格的最近邻格点连接关系
edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (2, 6)]

##### 3.计算节点坐标
#@ 定义六边形参数
a = 1.5  # 六边形边长
sqrt3 = np.sqrt(3) # 数学常数√3，因为六边形坐标计算经常用到

## 定义底部六边形坐标 
coords = {} # 创建空字典存储每个格点的坐标

# 底部六边形端点坐标(以位于六边形左中的格点1为基准原点)
coords[0] = (a/2, -sqrt3*a/2) # 左下
coords[1] = (0, 0) # 左中
coords[2] = (a/2, sqrt3*a/2)  # 左上
coords[4] = (3*a/2, -sqrt3*a/2)  # 右下
coords[5] = (2*a, 0)  # 右中
coords[6] = (3*a/2, sqrt3*a/2)  # 右上

# 上方新六边形的中间点
coords[3] = (0, sqrt3*a)  # 连接0和3
coords[7] = (2*a, sqrt3*a)  # 连接4和7

##### 4.创建图形
fig, ax = plt.subplots(figsize=(10, 10)) # plt.subplots()：Matplotlib的核心函数，用于创建图形窗口fig(可以包含多个坐标轴)和坐标轴ax

#### 4.1绘制honeycomb晶格连接线
for i, j in edges: # 提取每个最近邻格点i,j
    x1, y1 = coords[i] # 点1的横纵坐标
    x2, y2 = coords[j] # 点2的横纵坐标
    ax.plot([x1, x2], [y1, y2],  # 在坐标轴ax上绘制连接点1到点2的线段
            color='gray',        # 灰色
            linewidth=2,         # 线宽2点
            alpha=0.7,           # 透明度
            zorder=1)            # zorder的值为图层顺序，数值越大，绘制在越上层，此处1为在底层绘制

#### 4.2绘制格点
node_colors = [] # 创建列表存储每个格点的颜色值

### 通过循环遍历8个格点来绘制
for i in range(8): 
    x, y = coords[i] # 从坐标字典中获取当前格点的(x, y)坐标
    
    ## 使用密度值映射颜色
    color_val = (densities[i] - min(densities)) / (max(densities) - min(densities)) # color_val计算原因：将密度值归一化到[0, 1]区间
    node_color = plt.cm.RdYlBu_r(color_val) # 使用颜色映射函数plt.cm.RdYlBu_r()获取颜色(_r表示反向),即color_val值从小到大对应颜色为蓝-黄-红
    node_colors.append(node_color) # 将当前格点的颜色添加到列表中
    
    ## 绘制圆形格点
    circle = patches.Circle((x, y),   # patches.Circle()用于在图表上创建一个圆形的图形元素，其中(x, y)代表圆心的坐标
                            radius=0.12,            # 圆半径
                            facecolor=node_color,  # 圆形填充颜色
                            edgecolor='black',     # 圆形边框颜色选为black(黑色)
                            linewidth=2,           # 圆形边框宽度
                            zorder=3)              # 绘制图层级数为3，确保格点绘制在连接线（zorder=1）之上，但低于文本标签（zorder=4）
    ax.add_patch(circle) # 将圆形添加到坐标轴
    
    ## 添加格点编号
    ax.text(x, y, str(i),        # ax.text(x,y,str())是 Matplotlib 中用于在图表位置(x, y)中添加str()括号里的文本的核心函数。
            ha='center',         # 水平对齐位置(center为居中)
            va='center',         # 垂直对齐位置(center为居中)
            fontsize=15,         # 字体大小
            fontweight='heavy',  # 字体粗细(heavy为很粗的字体)
            color='black',       # 字体颜色
            zorder=4)
    
    ## 给每个格点添加密度值标签
    # 根据节点位置调整标签位置
    if i in [0, 4, 1, 5]:    # 中下方的4个格点
        label_y = y - 0.3    # 标签位置的纵坐标为：y - 0.3
    else:                    # 上方的4个格点
        label_y = y + 0.3    # 标签位置的纵坐标为：y + 0.3
    
    ax.text(x, label_y, f'{densities[i]:.5f}', 
            ha='center', va='center', fontsize=15, 
            bbox=dict(        # bbox=dict(...)：背景框样式,即为文本添加背景框
                boxstyle="round,pad=0.2",   # 框样式，此处为圆角矩形，内边距0.2
                facecolor="lightyellow",    # 背景框填充颜色，此处为浅黄色
                edgecolor="gray",           # 边框颜色，此处为灰色
                alpha=0.8),                 # 透明度
            zorder=2)

### 4.3设置坐标轴
all_x = [coords[i][0] for i in range(8)]    # 获取所有格点的x坐标
all_y = [coords[i][1] for i in range(8)]    # 获取所有格点的y坐标
ax.set_xlim(min(all_x)-0.5, max(all_x)+0.5) # 设置x轴的显示范围
ax.set_ylim(min(all_y)-0.7, max(all_y)+0.7) # 设置y轴的显示范围
ax.set_aspect('equal') # set_aspect()函数用于设置坐标轴的纵横比（宽高比）,而'equal'为1:1比例(为了保持honeycomb形状)
ax.axis('off') # 完全关闭坐标轴的显示

### 4.4添加标题
ax.set_title('Honeycomb Lattice with 8 Sites and Density Values', fontsize=25, pad=8) # pad：标题与图形的间距

### 4.5创建颜色条
sm = plt.cm.ScalarMappable(  # plt.cm.ScalarMappable()：创建可映射到颜色的对象，用于连接颜色映射和数值范围与生成颜色条提供数据
    cmap=plt.cm.RdYlBu_r,    # 表示从红(低值) → 黄(中值) → 蓝(高值)的颜色渐变
    norm=plt.Normalize(      # Normalizer(a,b)为归一化器,将处于[a,b]区间的原始数据值线性映射到[0, 1]区间,计算方法为上面变量color_val的计算
        min(densities), max(densities))
)
# set_array()步骤必须存在，要给ScalarMappable提供实际数据！！！
sm.set_array([]) # 此处设置空数组是只需要颜色条的显示功能，不需要实际的数据映射，从而避免无意义的数据复制
cbar = plt.colorbar(          # 创建颜色条
    sm,                       # ScalarMappable对象，用于提供颜色映射数据
    ax=ax,                    # 确定颜色条关联的坐标轴，并将颜色条显示在指定坐标轴旁
    orientation='vertical',   # 颜色条方向，此处'vertical'为垂直
    fraction=0.046,           # 颜色条粗细：占坐标轴高度的比例
    pad=0.04                  # 间距：颜色条与坐标轴的间距
)
cbar.ax.tick_params(    # cbar.ax.tick_params()：精细控制颜色条的刻度样式(其中cbar.ax是颜色条内部的坐标轴对象)
    axis='y',           # 设置y轴刻度(对应垂直颜色条)
    labelsize=15,       # 刻度字体大小
    length=6,           # 刻度线长度
    width=1,            # 刻度线宽度
    pad=8               # 标签与刻度线的距离
)
cbar.set_label('Density Value', fontsize=18, labelpad=15) # 设置颜色条标签，其中labelpad为标签与颜色条的距离

plt.tight_layout()
plt.show()

In [ ]:
### DMRG计算结果
# 输入数据
data_n = """
[
  [[0], [0.7407134066834603,              0]],
  [[1], [0.8768596904100008,              0]],
  [[2], [0.7407134066834602,              0]],
  [[3], [0.8768596904100008,              0]],
  [[4], [0.6374239769233427,              0]],
  [[5], [0.7450029259831956,              0]],
  [[6], [0.6374239769233425,              0]],
  [[7], [0.7450029259831961,              0]]
]
"""
data_Sz = """
[
  [[0], [-0.00715024677042,              0]],
  [[1], [0.131494100875,              0]],
  [[2], [-0.00583798790502,              0]],
  [[3], [ 0.13149410561,              0]],
  [[4], [-0.00583796042653,              0]],
  [[5], [0.131494106954,              0]],
  [[6], [-0.00715022529182,              0]],
  [[7], [0.131494106954,              0]]
]
"""

# 提取出Sz值并储存
parsed_data_Sz = ast.literal_eval(data_Sz) # 通过ast.literal_eval安全将字符串data转换为实际的Python列表对象
Sz_DMRG = [item[1][0] for item in parsed_data_Sz] # 使用列表推导式从解析后的数据中提取特定值

# 提取出密度n值并储存
parsed_data_n = ast.literal_eval(data_n) 
n_DMRG = [item[1][0] for item in parsed_data_n] 

# 计算ED与DMRG之间的误差
Sz_relative_error = [] # 创建存储每个格点的S_z相对误差
n_relative_error = [] # 创建存储每个格点的n相对误差
Sz_absolute_error = [] # 创建存储每个格点的S_z绝对误差
n_absolute_error = [] # 创建存储每个格点的n绝对误差
for i in range(L):
    Sz_re_err = abs((Sz_ED_1[i] - Sz_DMRG[i])/Sz_ED_1[i])
    n_re_err = abs((n_ED_1[i] - n_DMRG[i])/n_ED_1[i])
    Sz_relative_error.append(Sz_re_err)
    n_relative_error.append(n_re_err)
    
    Sz_abs_err = abs(Sz_ED_1[i] - Sz_DMRG[i])
    n_abs_err = abs(n_ED_1[i] - n_DMRG[i])
    Sz_absolute_error.append(Sz_abs_err)
    n_absolute_error.append(n_abs_err)

# 总相对误差(即每个格点相对误差的和)
Sz_total_relative_error = sum(Sz_relative_error)
n_total_relative_error = sum(n_relative_error)
print(f'\n自旋总相对误差: {Sz_total_relative_error:.5e}')
print(f'密度总相对误差: {n_total_relative_error:.5e}')

# 总相对误差(即每个格点相对误差的和)
Sz_total_absolute_error = sum(Sz_absolute_error)
n_total_absolute_error = sum(n_absolute_error)
print(f'\n自旋总绝对误差: {Sz_total_absolute_error:.5e}')
print(f'密度总绝对误差: {n_total_absolute_error:.5e}')


#---------------------------------------------------------------------------------------------------------------------
#---------------------------------########### 自旋Sz算符平均值 ################---------------------------------------
print()
print('='*80)
print('每个格点的自旋平均值 <S^z_i>：')
print()

## 可视化S_z平均值
plt.figure(figsize=(10, 5))
plt.plot(range(L), Sz_ED_1, 'bo-', linewidth=2, markersize=8, label="ED")
plt.plot(range(L), Sz_DMRG, 'ys-', linewidth=2, markersize=8, label="DMRG")
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('$\\langle S_i^z \\rangle$', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('Average spin per site $\\langle S_i^z \\rangle$', fontsize=20)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=13)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

## 可视化S_z相对误差
plt.figure(figsize=(10, 5))
plt.plot(range(L), Sz_absolute_error, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('Sz absolute error', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('The absolute error of average spin per site', fontsize=20)
plt.grid(True, alpha=0.3)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

plt.show()



#--------------------------------------------------------------------------------------------------------------------
#---------------------------------########### 密度算符平均值 ################----------------------------------------
print()
print('='*80)
print('每个格点的密度平均值 <n_i>：')
print()

## 可视化n平均值
plt.figure(figsize=(10, 5))
plt.plot(range(L), n_ED_1, 'bo-', linewidth=2, markersize=8, label="ED")
plt.plot(range(L), n_DMRG, 'ys-', linewidth=2, markersize=8, label="DMRG")
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('$\\langle n_i \\rangle$', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('Average density per site $\\langle n_i \\rangle$', fontsize=20)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=13)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

## 可视化密度n相对误差
plt.figure(figsize=(10, 5))
plt.plot(range(L), n_absolute_error, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Site index i', fontsize=15)
plt.ylabel('n absolute error', fontsize=15) 
# 注：$符号用作数学模式的分界符，即表明$...$之间的内容是需要用特殊规则排版的数学公式(以LaTeX为标准)。其中\\langle ...\\rangle表示期望值
plt.title('The absolute error of average density per site', fontsize=20)
plt.grid(True, alpha=0.3)

# 调节x轴刻度。其中plt.gca()获取当前活动的坐标轴对象；.xaxis获取x轴对象，使得可以专门设置x轴的属性
plt.gca().xaxis.set_major_locator(MultipleLocator(1))  # 主刻度间隔为1[其中set_major_locator()设置主刻度定位器]
plt.gca().xaxis.set_minor_locator(MultipleLocator(0.5))  # 次刻度间隔为0.5[其中set_minor_locator()设置次刻度定位器]

plt.show()